# Learner Activity: FFT, Regression, And Prophet

Complete the missing code to identify repeating cycles, model the strongest cycle with a lightweight regression, and forecast daily revenue with Prophet.


## Setup


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from prophet import Prophet


ACTIVITY_DATA_DIR = Path("Activity Data")

FFT_DATA_FILE = ACTIVITY_DATA_DIR / "fft-sample-data.csv"
PROPHET_DATA_FILE = ACTIVITY_DATA_DIR/ "prophet-sample-data.csv"

DATA_FILE = ACTIVITY_DATA_DIR / "fft-sample-data.csv"
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing data file: {DATA_FILE}")


## 1. FFT: Find Repeating Transaction Cycles

Load daily transaction data and use FFT to identify dominant repeating periods.


In [6]:
# Load the FFT dataset from FFT_DATA_FILE.
# Parse the date column so Plotly treats it as a date.
fft_df = pd.read_csv(ACTIVITY_DATA_DIR / "fft-sample-data.csv")
fft_df.head()


,date,transactions
0,2025-01-01,250.09
1,2025-01-02,266.98
2,2025-01-03,273.58
3,2025-01-04,272.62
4,2025-01-05,272.28


In [7]:
# Plot daily transactions over time.
# Use date on the x-axis and transactions on the y-axis.
fig = px.line(
    fft_df,
    x="date",
    y="transactions",
    title="FFT Example: Daily Transactions Over Time",
)
fig.update_layout(template="plotly_white")
fig.show()


In [8]:
# Convert transactions into a numeric signal.
signal = fft_df["transactions"].to_numpy(dtype=float)

# Center the signal by subtracting its mean.
centered_signal = signal - signal.mean()

# Calculate FFT values from the centered signal.
fft_values = np.fft.rfft(centered_signal)

# Calculate the matching frequencies.
frequencies = np.fft.rfftfreq(len(centered_signal), d=1)

# Convert FFT values to magnitudes.
magnitudes = np.abs(fft_values)

# Exclude the zero-frequency component (index 0)
frequency_values = frequencies[1:]
magnitude_values = magnitudes[1:]

# Build spectrum dataframe
spectrum_df = pd.DataFrame({
    "frequency": frequency_values,
    "magnitude": magnitude_values,
})

# Convert frequency → period (days)
spectrum_df["period_days"] = 1 / spectrum_df["frequency"]

# Sort by strongest signals
top_components = spectrum_df.sort_values("magnitude", ascending=False).head()
top_components[["period_days", "magnitude"]].round(2)

,period_days,magnitude
11,15.0,2036.87
0,180.0,943.60
3,45.0,820.79
19,9.0,627.11
1,90.0,455.58


In [9]:
# Plot magnitude by period in days.
# The highest peaks show the strongest repeating cycles.
fig = px.line(
    spectrum_df,
    x="period_days",
    y="magnitude",
    title="FFT Spectrum: Dominant Periods",
)
fig.update_layout(template="plotly_white", xaxis_title="Period (days)")
fig.show()


### FFT Interpretation

Write 2-3 sentences: What are the strongest repeating periods? Do they look weekly, monthly, or longer-term?


## 2. FFT-Informed Simple Regression Recap

Reuse the simple linear regression function from Week 2 Day 1. Use the strongest FFT period to create one seasonal sine feature, then regress transactions on that seasonal feature.


In [10]:
def simple_linear_regression(x: np.ndarray, y: np.ndarray) -> dict:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x_mean = x.mean()
    y_mean = y.mean()

    slope = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
    intercept = y_mean - (slope * x_mean)
    predictions = intercept + (slope * x)

    residuals = y - predictions
    sse = np.sum(residuals ** 2)
    sst = np.sum((y - y_mean) ** 2)
    r_squared = 1 - (sse / sst)
    rmse = np.sqrt(np.mean(residuals ** 2))

    return {
        "slope": slope,
        "intercept": intercept,
        "predictions": predictions,
        "sse": sse,
        "sst": sst,
        "r_squared": r_squared,
        "rmse": rmse,
    }

# Use the strongest FFT period as the seasonal period for a regression feature.
# Pick the first row from top_components.
dominant_period = top_components.iloc[0][
    "period_days"
]

regression_df = fft_df.copy()

# Create a time index from 0 to the number of rows minus 1.
regression_df["t"] = np.arange(len(regression_df))

# Create one sine-wave seasonal feature using the dominant period.
regression_df["seasonal_signal"] = np.sin(
    2 * np.pi * regression_df["t"] / dominant_period
)

# Identify the regression x values and y values.
x = regression_df["seasonal_signal"].to_numpy(dtype=float)
y = regression_df["transactions"].to_numpy(dtype=float)

# Reuse yesterday's simple regression function.
regression_result = simple_linear_regression(x, y)

# Store the fitted values from the regression result.
regression_df["regression_fitted"] = regression_result["predictions"]

# Build a readable summary of the regression result.
regression_summary = {
    "dominant_period_days": dominant_period,
    "slope": regression_result["slope"],
    "intercept": regression_result["intercept"],
    "rmse": regression_result["rmse"],
    "r_squared": regression_result["r_squared"],
}
pd.Series(regression_summary)


dominant_period_days     15.000000
slope                    22.630951
intercept               266.117111
rmse                     12.706591
r_squared                 0.613310
dtype: float64

In [11]:
# Plot actual transactions against the FFT-informed simple regression curve.
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=regression_df["t"],
        y=regression_df["transactions"],
        mode="lines",
        name="Actual",
    )
)
fig.add_trace(
    go.Scatter(
        x=regression_df["t"],
        y=regression_df["regression_fitted"],
        mode="lines",
        name="Seasonal regression fitted",
    )
)
fig.update_layout(
    title="FFT-Informed Simple Regression Fit",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Transactions",
)
fig.show()


### Regression Interpretation

Write 2-3 sentences: Does the single seasonal regression feature capture the dominant up-and-down cycle? What movement does it miss because it only uses one predictor?

ANSWER: Yes, the single seasonal regression does capture the dominant up-and-down cycle pattern in the data. However, it misses other movements like smaller cycles, sudden spikes, and any trend changes overtime.

## 3. Prophet: Packaged Additive Forecasting

Fit a Prophet model and forecast the next 60 days.


In [12]:
# Load Prophet data from PROPHET_DATA_FILE.
# Prophet expects columns named ds and y.
prophet_df = pd.read_csv(PROPHET_DATA_FILE, parse_dates=["ds"])
prophet_df.head()


,ds,y
0,2024-01-01,778.63
1,2024-01-02,797.96
2,2024-01-03,807.44
3,2024-01-04,783.70
4,2024-01-05,745.50


In [13]:
# Create the Prophet model.
# Turn on weekly and yearly seasonality, but leave daily seasonality off.
model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
)

# Fit the model using prophet_df.
model.fit(prophet_df)

# Forecast 60 days into the future.
future = model.make_future_dataframe(periods=60)
forecast = model.predict(future)

# Display the most useful forecast columns.
forecast_columns = [
    "ds",
    "yhat",
    "yhat_lower",
    "yhat_upper",
    "trend",
    "weekly",
    "yearly",
]
forecast[forecast_columns].tail()


12:41:28 - cmdstanpy - INFO - Chain [1] start processing


12:41:28 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper,trend,weekly,yearly
475,2025-04-20,546.986182,526.673133,565.291664,565.591894,-23.234891,4.629179
476,2025-04-21,569.354250,550.447661,586.229777,565.197240,-0.384465,4.541475
477,2025-04-22,590.770469,572.370877,610.572456,564.802585,21.563414,4.404469
478,2025-04-23,597.513937,579.194923,615.768668,564.407931,28.884313,4.221693
479,2025-04-24,581.361676,563.347796,599.767948,564.013276,13.350986,3.997413


In [14]:
# Plot actual data, forecast, and uncertainty interval.
forecast_fig = go.Figure()
forecast_fig.add_trace(
    go.Scatter(x=prophet_df["ds"], y=prophet_df["y"], mode="lines", name="Actual")
)
forecast_fig.add_trace(
    go.Scatter(x=forecast["ds"], y=forecast["yhat"], mode="lines", name="Forecast")
)
forecast_fig.add_trace(
    go.Scatter(
        x=forecast["ds"],
        y=forecast["yhat_upper"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    )
)
forecast_fig.add_trace(
    go.Scatter(
        x=forecast["ds"],
        y=forecast["yhat_lower"],
        mode="lines",
        fill="tonexty",
        line=dict(width=0),
        name="Prediction Interval",
        hoverinfo="skip",
    )
)
forecast_fig.update_layout(
    title="Prophet Forecast: Daily Revenue",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Revenue",
)
forecast_fig.show()


In [15]:
# Plot trend, weekly seasonality, and yearly seasonality components.
components_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=False,
    subplot_titles=("Trend", "Weekly Seasonality", "Yearly Seasonality"),
)
components_fig.add_trace(
    go.Scatter(x=forecast["ds"], y=forecast["trend"], mode="lines", name="Trend"),
    row=1,
    col=1,
)
components_fig.add_trace(
    go.Scatter(x=forecast["ds"], y=forecast["weekly"], mode="lines", name="Weekly"),
    row=2,
    col=1,
)
components_fig.add_trace(
    go.Scatter(x=forecast["ds"], y=forecast["yearly"], mode="lines", name="Yearly"),
    row=3,
    col=1,
)
components_fig.update_layout(
    title="Prophet Components",
    template="plotly_white",
    height=800,
    showlegend=False,
)
components_fig.show()


In [16]:
# Summarize the forecast.
# Use the final 30 forecast rows and calculate the average yhat value.
next_30 = forecast.tail()["yhat"].mean()

print("Prophet Results")
print("---------------")
print("Forecast horizon: 60 days")
print(f"Average forecast for next 30 days: {next_30:.2f}")
print(f"Last observed date: {prophet_df['ds'].max().date()}")
print(f"Last forecast date: {forecast['ds'].max().date()}")


Prophet Results
---------------
Forecast horizon: 60 days
Average forecast for next 30 days: 577.20
Last observed date: 2025-02-23
Last forecast date: 2025-04-24


### Prophet Interpretation

Write 2-3 sentences: Is the forecast trending upward, downward, or mostly stable? Which component explains regular within-week movement?
- ANSWER: The forecast is mostly stable with a slight upward or downward trend. This indicate a gradual long-term change rather than sharp growth or decline. The component that explains the regular within-week movement is the weekly seasonality component that captures specific patterns such as higher or lower revenue on specific days of the week.